<a href="https://colab.research.google.com/github/1021114Carlos/MIT_MM_Finance/blob/Finance-shop/Courses/Derivative_Markets/M7_exotic_options_numerical_methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Library

In [ ]:
from IPython.display import display, Math
import math
from dataclasses import dataclass
from typing import Dict, Tuple
import matplotlib.pyplot as plt
import networkx as nx

(Q1)

In [ ]:
print("Risk Neutral Probability")
display(Math(r"q = \frac{(1 + r) - d}{u - d}"))

print("\nC(j;i)\n")
display(Math(r"c(j;i) = \max(0, \ \beta(qC(up \ next; i + 1) + (1-q)C(down \ next; i + 1))-F)"))


In [ ]:
"""
Extendible (renewable) call option on a 3-period binomial tree with a per-period fee F.

At each node BEFORE expiration, the holder chooses the best of:
(i)  expire now: 0
(ii) exercise now: max(S - K, 0)
(iii) pay fee and continue: PV_rn( next node option values ) - F

This script:
- asks user for inputs (S0, K, r, F) and optional u, d, N
- builds the recombining tree of stock prices
- prices the extendible call by backward induction
- prints a table of C(j;i)
- plots the node network (tree) with labels for S and C

Requires: matplotlib, networkx
Install if needed:
    pip install matplotlib networkx
"""

import math
from dataclasses import dataclass
from typing import Dict, Tuple

import matplotlib.pyplot as plt
import networkx as nx


@dataclass
class Params:
    S0: float
    K: float
    r: float          # simple compounding per period
    F: float          # fee paid at beginning of each period BEFORE expiration
    u: float = 2.0    # up multiplier (up by 100% => 2.0)
    d: float = 0.5    # down multiplier (down by 50% => 0.5)
    N: int = 3        # number of periods until expiration


def get_float(prompt: str, default: float) -> float:
    s = input(f"{prompt} [default {default}]: ").strip()
    return default if s == "" else float(s)


def get_int(prompt: str, default: int) -> int:
    s = input(f"{prompt} [default {default}]: ").strip()
    return default if s == "" else int(s)


def build_stock_tree(p: Params) -> Dict[Tuple[int, int], float]:
    """
    Node indexing: (i, j) where
      i = time step, i=0..N
      j = node index at time i, j=1..(i+1), top-to-bottom
    At time i, node j has:
      ups = i - (j-1)
      downs = (j-1)
      S(i,j) = S0 * u^ups * d^downs
    """
    S = {}
    for i in range(p.N + 1):
        for j in range(1, i + 2):
            ups = i - (j - 1)
            downs = (j - 1)
            S[(i, j)] = p.S0 * (p.u ** ups) * (p.d ** downs)
    return S


def price_extendible_call(p: Params, S: Dict[Tuple[int, int], float]) -> Tuple[Dict[Tuple[int, int], float], float]:
    """
    Backward induction with three-way max at each node:
      value = max(0, exercise, continue)
    where:
      exercise = max(S - K, 0)
      continue = (q * C(up_child) + (1-q) * C(down_child)) / (1+r) - F   for i < N
    At maturity i=N:
      C = max(S - K, 0)
    """
    # Risk-neutral probability
    q = ((1 + p.r) - p.d) / (p.u - p.d)

    if not (0.0 <= q <= 1.0):
        raise ValueError(
            f"Risk-neutral q={q:.6f} is not in [0,1]. Check inputs: u={p.u}, d={p.d}, r={p.r}."
        )

    C = {}

    # Terminal values (time N)
    i = p.N
    for j in range(1, i + 2):
        C[(i, j)] = max(S[(i, j)] - p.K, 0.0)

    # Backward induction for i = N-1 down to 0
    for i in range(p.N - 1, -1, -1):
        for j in range(1, i + 2):
            exercise = max(S[(i, j)] - p.K, 0.0)

            # children at next time:
            up_child = (i + 1, j)       # same j is "up"
            dn_child = (i + 1, j + 1)   # j+1 is "down"

            cont = (q * C[up_child] + (1 - q) * C[dn_child]) / (1 + p.r) - p.F

            C[(i, j)] = max(0.0, exercise, cont)

    return C, q


def print_C_table(p: Params, S: Dict[Tuple[int, int], float], C: Dict[Tuple[int, int], float]) -> None:
    print("\n=== Node values (S and C) ===")
    for i in range(0, p.N + 1):
        print(f"\nTime i={i}")
        for j in range(1, i + 2):
            print(f"  Node (j={j}; i={i}):  S={S[(i,j)]:.6f}   C={C[(i,j)]:.6f}")


def plot_tree(p: Params, S: Dict[Tuple[int, int], float], C: Dict[Tuple[int, int], float], q: float) -> None:
    """
    Draw a directed tree with labels showing S and C at each node.
    """
    G = nx.DiGraph()

    # Add nodes and edges
    for i in range(0, p.N + 1):
        for j in range(1, i + 2):
            G.add_node((i, j))

            if i < p.N:
                G.add_edge((i, j), (i + 1, j), prob=q)         # up
                G.add_edge((i, j), (i + 1, j + 1), prob=1 - q)  # down

    # Positions for a "tree" layout: x=time, y=-(j-1) + i/2 to spread
    pos = {}
    for i in range(0, p.N + 1):
        for j in range(1, i + 2):
            # center vertically by shifting by i/2
            pos[(i, j)] = (i, -(j - 1) + i / 2)

    # Node labels: show S and C
    labels = {}
    for i in range(0, p.N + 1):
        for j in range(1, i + 2):
            labels[(i, j)] = f"(j={j};i={i})\nS={S[(i,j)]:.2f}\nC={C[(i,j)]:.2f}"

    plt.figure(figsize=(10, 6))
    nx.draw_networkx_nodes(G, pos, node_size=1600)
    nx.draw_networkx_edges(G, pos, arrows=True, arrowstyle="-|>", arrowsize=14)
    nx.draw_networkx_labels(G, pos, labels, font_size=8)

    # Edge labels: show q and 1-q
    edge_labels = {}
    for (u, v, data) in G.edges(data=True):
        edge_labels[(u, v)] = f"{data['prob']:.4f}"
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)

    plt.title(
        f"3-Period Extendible Call Tree (Fee F={p.F}, r={p.r}, q={q:.4f})\n"
        f"At each node: max(0, S-K, discounted RN continuation - F)"
    )
    plt.axis("off")
    plt.tight_layout()
    plt.show()


def main():
    print("Extendible (renewable) Call on a Binomial Tree")
    print("Enter values; press Enter to accept the default shown in brackets.\n")

    S0 = get_float("S0 (current stock price)", 100.0)
    K  = get_float("K (strike)", 80.0)
    r  = get_float("r (simple interest per period, e.g. 0.2518)", 0.2518)
    F  = get_float("F (fee paid at t=0,1,2... before expiration)", 20.0)

    # Optional: let user override u,d,N
    u  = get_float("u (up multiplier)", 2.0)
    d  = get_float("d (down multiplier)", 0.5)
    N  = get_int("N (number of periods)", 3)

    p = Params(S0=S0, K=K, r=r, F=F, u=u, d=d, N=N)

    S = build_stock_tree(p)
    C, q = price_extendible_call(p, S)

    print(f"\nRisk-neutral probability q = {q:.6f}")
    print_C_table(p, S, C)

    # Show requested C(j;i) range:
    print("\n=== Requested C(j;i) for i=1,2,3 and j=1..4 (N/A if node doesn't exist) ===")
    for i in [1, 2, 3]:
        row = []
        for j in [1, 2, 3, 4]:
            if (i, j) in C:
                row.append(f"C({j};{i})={C[(i,j)]:.6f}")
            else:
                row.append(f"C({j};{i})=N/A")
        print("  " + " | ".join(row))

    plot_tree(p, S, C, q)


if __name__ == "__main__":
    main()


(Q2)

In [ ]:
def norm_cdf(x: float):
    # Standard normal CDF using error function
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def margrabe_exchange_option(Au: float, Ag: float, beta: float,
                              T: float, sigma_Au: float,
                             sigma_Ag: float, rho: float, q_Au: float = 0.0,
                             q_Ag: float = 0.0,   ):
    """
    Price of European option to exchange beta units of S for 1 unit of G at time T:
        payoff = max(G_T - beta*S_T, 0)

    Margrabe formula with yields:
        V0 = G0*e^{-q_G T} N(d1) - beta*S0*e^{-q_S T} N(d2)
        sigma_eff = sqrt(sigma_G^2 + sigma_S^2 - 2*rho*sigma_G*sigma_S)
        d1 = [ln(G0/(beta*S0)) + (q_S - q_G + 0.5*sigma_eff^2)T] / (sigma_eff*sqrt(T))
        d2 = d1 - sigma_eff*sqrt(T)
    """
    X0 = beta * S0
    sigma_eff = math.sqrt(sigma_Au**2 + sigma_Ag**2 - 2.0 * rho * sigma_Au * sigma_Ag)

    if sigma_eff <= 0 or T <= 0:
        raise ValueError("Need T>0 and sigma_eff>0. Check inputs (vols/correlation).")

    d1 = (math.log(G0 / X0) + (q_Ag - q_Au + 0.5 * sigma_eff**2) * T) / (sigma_eff * math.sqrt(T))
    d2 = d1 - sigma_eff * math.sqrt(T)

    V0 = (G0 * math.exp(-q_Au * T) * norm_cdf(d1)
          - X0 * math.exp(-q_Ag * T) * norm_cdf(d2))
    return V0

def get_float(prompt: str, default: float) -> float:
    s = input(f"{prompt} [default {default}]: ").strip()
    return default if s == "" else float(s)

if __name__ == "__main__":
    print("European Exchange Option: give beta oz silver for 1 oz gold in T years\n")

    Au0 = get_float("Gold price G0 (per oz)", 1520.0)
    Ag0 = get_float("Silver price S0 (per oz)", 16.0)
    beta = get_float("beta (oz of silver to give)", 100.0)
    T = get_float("T (years)", 1.0)

    # r included for completeness, but not needed unless you also model carry via q_G/q_S
    r = get_float("r (risk-free, cont comp). Not used unless you set carries.", 0.0837)

    sigma_Au = get_float("sigma_G (gold vol)", 0.1814)
    sigma_Ag = get_float("sigma_S (silver vol)", 0.1814)
    rho = get_float("rho (correlation)", 0.53)

    q_Au = get_float("q_G (gold yield/carry, default 0)", 0.0)
    q_Ag = get_float("q_S (silver yield/carry, default 0)", 0.0)

    price = margrabe_exchange_option(Au0, Ag0, beta, T, sigma_Au, sigma_Ag, rho, q_Au, q_Ag)
    print(f"\nOption price V0 = {price:.4f}")
